# Music-Sync Log Analysis

This notebook loads the `music-sync` logs from `/var/log/music-sync` and computes statistics about the most numerous request-related entries.

In [ ]:
# Section 1: Import Libraries and Setup
import os
import re
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

log_dir = Path("/var/log/music-sync")

In [ ]:
# Section 2: Load and Inspect Log Files

# list files in the log directory (read_file may bypass permissions)
print("log files:", log_dir.glob("*") if log_dir.exists() else "none")

# read first few lines of the sample log using open()
sample = None
if log_dir.exists():
    for p in log_dir.glob("*.log*"):
        try:
            with open(p, "r", errors="ignore") as f:
                lines = [next(f) for _ in range(10)]
            sample = (p, lines)
            break
        except Exception as e:
            continue

print("sample file and lines:", sample)

In [ ]:
# Section 3: Parse Log Entries

# We'll parse each line and attempt to extract a "phase" or "request" keyword.
entry_re = re.compile(r"\[PH:([^\]]+)\]")

parsed = []
if log_dir.exists():
    for p in log_dir.glob("*.log*"):
        try:
            with open(p, "r", errors="ignore") as f:
                for line in f:
                    m = entry_re.search(line)
                    if m:
                        parsed.append(m.group(1))
        except Exception:
            pass

print(f"parsed {len(parsed)} entries")

In [ ]:
# Section 4: Count Request Types

counter = Counter(parsed)

# show top 20 phases
for phase, cnt in counter.most_common(20):
    print(f"{cnt:7} {phase}")


In [ ]:
# Section 5: Visualize Top Requests

most = counter.most_common(10)
if most:
    phases, counts = zip(*most)
    plt.figure(figsize=(8,4))
    plt.barh(phases[::-1], counts[::-1])
    plt.title("Top 10 Phases in music-sync logs")
    plt.xlabel("Occurrences")
    plt.show()
else:
    print("No phases to plot.")